In [ ]:
import csv
import datetime
import io
import re
import time
import logging
import argparse
from pathlib import Path
 
# Scraper dependencies — only needed when --scrape is passed
try:
    import requests
    from bs4 import BeautifulSoup
    _SCRAPER_AVAILABLE = True
except ImportError:
    _SCRAPER_AVAILABLE = False
 
log = logging.getLogger(__name__)
 
# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────
 
DAYS = {0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri", 5: "Sat", 6: "Sun"}
VENUE = "Alamo Drafthouse Woodbury"
 
CLOSURE_START = datetime.date(2024, 6, 6)
CLOSURE_END   = datetime.date(2024, 8, 26)
 
 
def d(y, m, day) -> datetime.date:
    return datetime.date(y, m, day)
 
 
def dstr(dt: datetime.date) -> str:
    return dt.strftime("%Y-%m-%d")
 
 
def dow(dt: datetime.date) -> str:
    return DAYS[dt.weekday()]
 
 
# ─────────────────────────────────────────────────────────────────────────────
# SHOWTIME LOGIC
# ─────────────────────────────────────────────────────────────────────────────
 
def showtimes(film_type: str, runtime: int, dt: datetime.date,
              release_dt: datetime.date) -> list[str]:
    """Return ordered showtime strings for one film on one date."""
    wday = dt.weekday()
    days_since = (dt - release_dt).days
    week = days_since // 7
 
    is_thu  = wday == 3
    is_fri  = wday == 4
    is_sat  = wday == 5
    is_sun  = wday == 6
    is_wknd = wday >= 4   # Fri / Sat / Sun
    is_wkdy = wday <= 2   # Mon / Tue / Wed
    long    = runtime >= 150
 
    times: list[str] = []
 
    if film_type == "BLOCK":
        if week == 0 and is_thu:
            times = ["7:00 PM"]
        elif week == 0 and is_fri:
            times = (["11:15 AM", "3:00 PM", "7:00 PM"] if long
                     else ["11:15 AM", "2:45 PM", "6:15 PM", "9:45 PM"])
        elif week == 0 and is_sat:
            times = (["11:00 AM", "3:00 PM", "7:00 PM"] if long
                     else ["11:15 AM", "2:45 PM", "6:15 PM", "9:45 PM"])
        elif week == 0 and is_sun:
            times = ["11:30 AM", "3:00 PM", "6:30 PM"]
        elif week == 0 and is_wkdy:
            times = ["3:45 PM", "7:15 PM"]
        elif week == 1 and is_fri:
            times = ["12:15 PM", "3:30 PM", "6:45 PM", "9:30 PM"]
        elif week == 1 and is_sat:
            times = ["12:00 PM", "3:15 PM", "6:45 PM", "9:30 PM"]
        elif week == 1 and is_sun:
            times = ["12:00 PM", "3:15 PM", "6:45 PM"]
        elif week == 1 and is_wkdy:
            times = ["3:30 PM", "7:00 PM"]
        elif week >= 2 and is_wknd:
            times = ["1:00 PM", "4:15 PM", "7:30 PM"]
        elif week >= 2 and is_wkdy:
            times = ["4:15 PM", "7:15 PM"]
 
    elif film_type == "STD":
        if week == 0 and is_thu:
            times = ["7:00 PM"]
        elif week <= 1 and is_wknd:
            times = ["12:30 PM", "3:30 PM", "6:45 PM"]
        elif week <= 1 and is_wkdy:
            times = ["3:45 PM", "7:00 PM"]
        elif week >= 2 and is_wknd:
            times = ["1:30 PM", "4:30 PM", "7:15 PM"]
        elif week >= 2 and is_wkdy:
            times = ["4:30 PM", "7:15 PM"]
 
    elif film_type == "ANIM":
        if week == 0 and is_thu:
            times = ["3:30 PM", "7:00 PM"]
        elif is_wknd:
            times = ["11:30 AM", "2:15 PM", "5:00 PM"]
        else:
            times = ["2:30 PM", "5:15 PM"]
 
    elif film_type == "HOLD":
        if is_wknd:
            times = ["1:30 PM", "4:30 PM"]
        else:
            times = ["4:30 PM"]
 
    elif film_type == "CLASS":
        if is_fri:
            times = ["4:15 PM", "7:15 PM"]
        elif is_sat:
            times = ["3:00 PM", "7:30 PM"]
        elif is_sun:
            times = ["2:00 PM", "5:00 PM"]
        else:
            times = ["7:00 PM"]
 
    # Long films: drop late-night shows
    if runtime >= 180:
        times = [t for t in times if not t.startswith(("9:", "10:"))]
 
    return times
 
 
# ─────────────────────────────────────────────────────────────────────────────
# FILM CATALOG
# Each entry: (title, year, release_date, end_date, type, runtime_mins)
#
# Types:
#   BLOCK = blockbuster / wide release (multiple showtimes per day)
#   STD   = standard new release (1 screen)
#   ANIM  = animation / family
#   HOLD  = holdover (reduced schedule)
#   CLASS = classic / revival / event (usually 1-2 shows)
# ─────────────────────────────────────────────────────────────────────────────
 
FILMS = [
    # ── 2021 ─────────────────────────────────────────────────────────────────
 
    # Classics / events (confirmed from Racket MN)
    ("Psycho (1960)",                          1960, d(2021,2,5),  d(2021,2,7),  "CLASS", 109),
    ("Speed Racer (2008)",                     2008, d(2021,4,9),  d(2021,4,11), "CLASS", 135),
    ("Clue (1985)",                            1985, d(2021,4,16), d(2021,4,18), "CLASS",  94),
    ("Point Break (1991)",                     1991, d(2021,6,11), d(2021,6,13), "CLASS", 122),
    ("Die Hard (1988)",                        1988, d(2021,8,6),  d(2021,8,8),  "CLASS", 132),
    ("The Texas Chain Saw Massacre (1974)",    1974, d(2021,10,8), d(2021,10,10),"CLASS",  84),
    ("Halloween (1978)",                       1978, d(2021,10,15),d(2021,10,17),"CLASS",  91),
    ("The Shining (1980)",                     1980, d(2021,10,22),d(2021,10,24),"CLASS", 144),
    ("A Nightmare on Elm Street (1984)",       1984, d(2021,10,29),d(2021,10,31),"CLASS",  91),
    ("Home Alone (1990)",                      1990, d(2021,12,3), d(2021,12,5), "CLASS", 103),
    ("Christmas Vacation (1989)",              1989, d(2021,12,10),d(2021,12,12),"CLASS",  97),
 
    # First-run / blockbusters
    ("Raya and the Last Dragon",               2021, d(2021,3,5),  d(2021,3,21), "ANIM",  107),
    ("Godzilla vs. Kong",                      2021, d(2021,4,2),  d(2021,4,18), "BLOCK", 113),
    ("Mortal Kombat (2021)",                   2021, d(2021,4,23), d(2021,5,9),  "BLOCK", 110),
    ("Demon Slayer: Mugen Train",              2021, d(2021,4,23), d(2021,5,9),  "ANIM",  117),
    ("Nobody (2021)",                          2021, d(2021,3,26), d(2021,4,11), "STD",    92),
    ("A Quiet Place Part II",                  2021, d(2021,5,28), d(2021,6,13), "BLOCK",  97),
    ("Cruella",                                2021, d(2021,5,28), d(2021,6,13), "BLOCK", 134),
    ("The Conjuring: The Devil Made Me Do It",2021, d(2021,6,4),  d(2021,6,20), "BLOCK", 112),
    ("F9: The Fast Saga",                      2021, d(2021,6,25), d(2021,7,18), "BLOCK", 143),
    ("Black Widow",                            2021, d(2021,7,9),  d(2021,7,25), "BLOCK", 134),
    ("Space Jam: A New Legacy",                2021, d(2021,7,16), d(2021,8,1),  "ANIM",  115),
    ("Jungle Cruise",                          2021, d(2021,7,30), d(2021,8,15), "BLOCK", 127),
    ("The Suicide Squad",                      2021, d(2021,8,6),  d(2021,8,22), "BLOCK", 132),
    ("Free Guy",                               2021, d(2021,8,13), d(2021,8,29), "BLOCK", 115),
    ("Candyman (2021)",                        2021, d(2021,8,27), d(2021,9,12), "BLOCK",  91),
    ("Shang-Chi and the Legend of the Ten Rings",2021,d(2021,9,3),d(2021,9,26),"BLOCK",132),
    ("No Time to Die",                         2021, d(2021,10,8), d(2021,10,31),"BLOCK", 163),
    ("Halloween Kills",                        2021, d(2021,10,15),d(2021,10,28),"BLOCK", 105),
    ("Dune (2021)",                            2021, d(2021,10,22),d(2021,11,18),"BLOCK", 155),
    ("The French Dispatch",                    2021, d(2021,10,22),d(2021,11,7), "STD",   108),
    ("Last Night in Soho",                     2021, d(2021,10,29),d(2021,11,11),"STD",   116),
    ("Eternals",                               2021, d(2021,11,5), d(2021,11,25),"BLOCK", 156),
    ("Ghostbusters: Afterlife",                2021, d(2021,11,19),d(2021,12,9), "BLOCK", 124),
    ("Encanto",                                2021, d(2021,11,24),d(2021,12,9), "ANIM",   99),
    ("Spider-Man: No Way Home",                2021, d(2021,12,17),d(2021,12,31),"BLOCK", 148),
    ("The Matrix Resurrections",               2021, d(2021,12,22),d(2021,12,31),"BLOCK", 148),
 
    # ── 2022 ─────────────────────────────────────────────────────────────────
 
    # Classics / events (confirmed from Racket MN)
    ("RoboCop (1987)",                         1987, d(2022,2,4),  d(2022,2,6),  "CLASS", 102),
    ("Raising Arizona (1987)",                 1987, d(2022,4,1),  d(2022,4,3),  "CLASS",  94),
    ("Suspiria (1977)",                        1977, d(2022,10,14),d(2022,10,16),"CLASS", 152),
    ("The Thing (1982)",                       1982, d(2022,10,21),d(2022,10,23),"CLASS", 109),
    ("Hellraiser (1987)",                      1987, d(2022,10,28),d(2022,10,30),"CLASS",  94),
 
    # First-run
    ("Spider-Man: No Way Home",                2021, d(2022,1,7),  d(2022,1,20), "HOLD",  148),
    ("Scream (2022)",                          2022, d(2022,1,14), d(2022,1,30), "BLOCK", 114),
    ("Jackass Forever",                        2022, d(2022,2,4),  d(2022,2,17), "STD",    96),
    ("Uncharted",                              2022, d(2022,2,18), d(2022,3,6),  "BLOCK", 116),
    ("The Batman",                             2022, d(2022,3,4),  d(2022,4,3),  "BLOCK", 176),
    ("The Lost City",                          2022, d(2022,3,25), d(2022,4,10), "STD",   112),
    ("Morbius",                                2022, d(2022,4,1),  d(2022,4,14), "STD",   104),
    ("Sonic the Hedgehog 2",                   2022, d(2022,4,8),  d(2022,4,24), "ANIM",  122),
    ("Everything Everywhere All at Once",      2022, d(2022,4,22), d(2022,6,5),  "STD",   139),
    ("Doctor Strange in the Multiverse of Madness",2022,d(2022,5,6),d(2022,5,26),"BLOCK",126),
    ("Top Gun: Maverick",                      2022, d(2022,5,27), d(2022,7,10), "BLOCK", 130),
    ("Jurassic World Dominion",                2022, d(2022,6,10), d(2022,7,3),  "BLOCK", 147),
    ("Lightyear",                              2022, d(2022,6,17), d(2022,7,3),  "ANIM",  100),
    ("Elvis (2022)",                           2022, d(2022,6,24), d(2022,7,17), "BLOCK", 159),
    ("Thor: Love and Thunder",                 2022, d(2022,7,8),  d(2022,7,31), "BLOCK", 119),
    ("Nope",                                   2022, d(2022,7,22), d(2022,8,7),  "BLOCK", 130),
    ("Bullet Train",                           2022, d(2022,8,5),  d(2022,8,18), "BLOCK", 126),
    ("DC League of Super-Pets",                2022, d(2022,7,29), d(2022,8,11), "ANIM",  105),
    ("Barbarian (2022)",                       2022, d(2022,9,9),  d(2022,9,22), "STD",   102),
    ("Don't Worry Darling",                    2022, d(2022,9,23), d(2022,10,6), "STD",   123),
    ("Halloween Ends",                         2022, d(2022,10,14),d(2022,10,27),"BLOCK", 111),
    ("Black Adam",                             2022, d(2022,10,21),d(2022,11,3), "BLOCK", 124),
    ("Black Panther: Wakanda Forever",         2022, d(2022,11,11),d(2022,12,4), "BLOCK", 161),
    ("Strange World",                          2022, d(2022,11,23),d(2022,12,4), "ANIM",  102),
    ("Violent Night",                          2022, d(2022,12,2), d(2022,12,15),"STD",   101),
    ("Avatar: The Way of Water",               2022, d(2022,12,16),d(2022,12,31),"BLOCK", 192),
    ("Puss in Boots: The Last Wish",           2022, d(2022,12,21),d(2022,12,31),"ANIM",  100),
 
    # ── 2023 ─────────────────────────────────────────────────────────────────
 
    # Classics / events (confirmed from Racket MN)
    ("Possession (1981)",                      1981, d(2023,5,5),  d(2023,5,7),  "CLASS", 124),
 
    # First-run
    ("Avatar: The Way of Water",               2022, d(2023,1,6),  d(2023,1,19), "HOLD",  192),
    ("Puss in Boots: The Last Wish",           2022, d(2023,1,6),  d(2023,1,19), "ANIM",  100),
    ("M3GAN (2022)",                           2022, d(2023,1,6),  d(2023,1,22), "BLOCK", 102),
    ("Knock at the Cabin",                     2023, d(2023,2,3),  d(2023,2,16), "STD",   100),
    ("Cocaine Bear",                           2023, d(2023,2,24), d(2023,3,9),  "BLOCK",  95),
    ("Ant-Man and the Wasp: Quantumania",      2023, d(2023,2,17), d(2023,3,9),  "BLOCK", 124),
    ("Creed III",                              2023, d(2023,3,3),  d(2023,3,23), "BLOCK", 116),
    ("Scream VI",                              2023, d(2023,3,10), d(2023,3,23), "BLOCK", 123),
    ("John Wick: Chapter 4",                   2023, d(2023,3,24), d(2023,4,13), "BLOCK", 169),
    ("The Super Mario Bros. Movie",            2023, d(2023,4,5),  d(2023,5,5),  "ANIM",   92),
    ("Evil Dead Rise",                         2023, d(2023,4,21), d(2023,5,4),  "BLOCK",  97),
    ("Guardians of the Galaxy Vol. 3",         2023, d(2023,5,5),  d(2023,5,25), "BLOCK", 150),
    ("Fast X",                                 2023, d(2023,5,19), d(2023,6,1),  "BLOCK", 141),
    ("The Little Mermaid (2023)",              2023, d(2023,5,26), d(2023,6,15), "BLOCK", 135),
    ("Spider-Man: Across the Spider-Verse",    2023, d(2023,6,2),  d(2023,6,22), "ANIM",  140),
    ("Indiana Jones and the Dial of Destiny",  2023, d(2023,6,30), d(2023,7,20), "BLOCK", 154),
    ("Mission: Impossible — Dead Reckoning Part One",2023,d(2023,7,12),d(2023,8,3),"BLOCK",163),
    ("Oppenheimer",                            2023, d(2023,7,21), d(2023,9,7),  "BLOCK", 180),
    ("Barbie (2023)",                          2023, d(2023,7,21), d(2023,9,7),  "BLOCK", 114),
    ("Blue Beetle",                            2023, d(2023,8,18), d(2023,8,31), "STD",   127),
    ("Teenage Mutant Ninja Turtles: Mutant Mayhem",2023,d(2023,8,4),d(2023,8,24),"ANIM", 99),
    ("Equalizer 3",                            2023, d(2023,9,1),  d(2023,9,14), "BLOCK", 109),
    ("The Nun II",                             2023, d(2023,9,8),  d(2023,9,21), "STD",   110),
    ("A Haunting in Venice",                   2023, d(2023,9,15), d(2023,9,28), "STD",   103),
    ("Saw X",                                  2023, d(2023,9,29), d(2023,10,12),"BLOCK", 118),
    ("The Creator (2023)",                     2023, d(2023,9,29), d(2023,10,12),"BLOCK", 133),
    ("The Exorcist: Believer",                 2023, d(2023,10,6), d(2023,10,19),"BLOCK", 111),
    ("Taylor Swift: The Eras Tour",            2023, d(2023,10,13),d(2023,11,2), "BLOCK", 168),
    ("Killers of the Flower Moon",             2023, d(2023,10,20),d(2023,11,9), "BLOCK", 206),
    ("Five Nights at Freddy's",                2023, d(2023,10,27),d(2023,11,9), "BLOCK", 110),
    ("The Marvels",                            2023, d(2023,11,10),d(2023,11,23),"BLOCK", 105),
    ("Wish (2023)",                            2023, d(2023,11,22),d(2023,12,7), "ANIM",   95),
    ("Hunger Games: The Ballad of Songbirds & Snakes",2023,d(2023,11,17),d(2023,12,7),"BLOCK",157),
    ("The Boy and the Heron",                  2023, d(2023,12,8), d(2023,12,28),"ANIM",  124),
    ("Aquaman and the Lost Kingdom",           2023, d(2023,12,22),d(2023,12,31),"BLOCK", 124),
    ("Migration (2023)",                       2023, d(2023,12,22),d(2023,12,31),"ANIM",   83),
 
    # ── 2024 PRE-CLOSURE (Jan 1 – Jun 5) ─────────────────────────────────────
 
    # Classics / events (confirmed from Racket MN)
    ("Happy Gilmore (1996)",                   1996, d(2024,3,8),  d(2024,3,8),  "CLASS",  92),
    ("Little Women (1994)",                    1994, d(2024,3,8),  d(2024,3,8),  "CLASS", 115),
    ("The Flintstones (1994)",                 1994, d(2024,3,8),  d(2024,3,8),  "CLASS", 100),
    ("Michael Clayton (2007)",                 2007, d(2024,3,8),  d(2024,3,8),  "CLASS", 119),
 
    # First-run
    ("Migration (2023)",                       2023, d(2024,1,5),  d(2024,1,14), "ANIM",   83),
    ("The Beekeeper",                          2024, d(2024,1,12), d(2024,1,25), "BLOCK", 105),
    ("Mean Girls (2024)",                      2024, d(2024,1,12), d(2024,1,18), "STD",   112),
    ("Argylle",                                2024, d(2024,2,2),  d(2024,2,15), "BLOCK", 139),
    ("Madame Web",                             2024, d(2024,2,14), d(2024,2,22), "STD",   116),
    ("Bob Marley: One Love",                   2024, d(2024,2,14), d(2024,2,22), "BLOCK", 107),
    ("Kung Fu Panda 4",                        2024, d(2024,3,8),  d(2024,3,21), "ANIM",   94),
    ("Dune: Part Two",                         2024, d(2024,3,1),  d(2024,4,4),  "BLOCK", 166),
    ("Ghostbusters: Frozen Empire",            2024, d(2024,3,22), d(2024,4,11), "BLOCK", 115),
    ("Civil War (2024)",                       2024, d(2024,4,12), d(2024,4,25), "BLOCK", 109),
    ("The First Omen",                         2024, d(2024,4,5),  d(2024,4,18), "STD",   120),
    ("The Fall Guy",                           2024, d(2024,5,3),  d(2024,5,16), "BLOCK", 126),
    ("Kingdom of the Planet of the Apes",      2024, d(2024,5,10), d(2024,5,23), "BLOCK", 145),
    ("IF (2024)",                              2024, d(2024,5,17), d(2024,5,30), "ANIM",  104),
    ("Furiosa: A Mad Max Saga",                2024, d(2024,5,24), d(2024,6,5),  "BLOCK", 148),
    ("The Garfield Movie",                     2024, d(2024,5,24), d(2024,6,5),  "ANIM",  101),
 
    # ── 2024 POST-REOPENING (Aug 27 onward) ──────────────────────────────────
 
    # Classics / events (confirmed from Racket MN)
    ("Blade (1998)",                           1998, d(2024,10,10),d(2024,10,10),"CLASS", 120),
    ("Deep Blue Sea (1999)",                   1999, d(2024,10,18),d(2024,10,18),"CLASS", 105),
    ("Tom Petty Heartbreakers Beach Party",    2024, d(2024,11,1), d(2024,11,1), "CLASS", 120),
 
    # First-run
    ("Alien: Romulus",                         2024, d(2024,8,16), d(2024,9,12), "BLOCK", 119),
    ("Beetlejuice Beetlejuice",                2024, d(2024,9,6),  d(2024,10,3), "BLOCK", 104),
    ("Transformers One",                       2024, d(2024,9,20), d(2024,10,3), "ANIM",  104),
    ("The Substance",                          2024, d(2024,9,20), d(2024,10,10),"STD",   141),
    ("Joker: Folie à Deux",                   2024, d(2024,10,4), d(2024,10,17),"BLOCK", 138),
    ("Terrifier 3",                            2024, d(2024,10,11),d(2024,10,24),"BLOCK", 125),
    ("Smile 2",                                2024, d(2024,10,18),d(2024,10,31),"BLOCK", 127),
    ("Venom: The Last Dance",                  2024, d(2024,10,25),d(2024,11,14),"BLOCK", 109),
    ("Heretic (2024)",                         2024, d(2024,11,8), d(2024,11,21),"BLOCK", 110),
    ("Gladiator II",                           2024, d(2024,11,22),d(2024,12,12),"BLOCK", 148),
    ("Wicked (2024)",                          2024, d(2024,11,22),d(2024,12,19),"BLOCK", 160),
    ("Moana 2",                                2024, d(2024,11,27),d(2024,12,19),"ANIM",  100),
    ("Nosferatu (2024)",                       2024, d(2024,12,25),d(2024,12,31),"BLOCK", 132),
    ("Sonic the Hedgehog 3",                   2024, d(2024,12,20),d(2024,12,31),"BLOCK", 109),
    ("Mufasa: The Lion King",                  2024, d(2024,12,20),d(2024,12,31),"ANIM",  118),
 
    # ── 2025 ─────────────────────────────────────────────────────────────────
 
    # Classics / events (confirmed from Racket MN)
    ("Phantom Thread (2017)",                  2017, d(2025,1,8),  d(2025,1,8),  "CLASS", 130),
    ("Speed Racer (2008)",                     2008, d(2025,1,8),  d(2025,1,8),  "CLASS", 135),
    ("Possession (1981)",                      1981, d(2025,1,8),  d(2025,1,9),  "CLASS", 124),
    ("A Clockwork Orange (1971)",              1971, d(2025,1,8),  d(2025,1,10), "CLASS", 136),
    ("The Killer (1989)",                      1989, d(2025,4,2),  d(2025,4,6),  "CLASS", 111),
    ("Castration Movie Anthology I. Traps",    2024, d(2025,4,2),  d(2025,4,4),  "CLASS",  90),
    ("Hush: The Shush Cut (2016)",             2016, d(2025,4,2),  d(2025,4,4),  "CLASS",  82),
    ("Coexistence, My Ass! (2025)",            2025, d(2025,4,2),  d(2025,4,6),  "CLASS",  90),
    ("City on Fire (1987)",                    1987, d(2025,4,2),  d(2025,4,6),  "CLASS", 106),
    ("The Best of Betty Boop",                 2024, d(2025,4,2),  d(2025,4,4),  "CLASS",  60),
    ("Mermaid (2026)",                         2026, d(2025,4,9),  d(2025,4,9),  "CLASS", 100),
 
    # First-run
    ("Den of Thieves 2: Pantera",              2025, d(2025,1,10), d(2025,1,23), "BLOCK", 130),
    ("Dog Man (2025)",                         2025, d(2025,1,17), d(2025,1,30), "ANIM",  110),
    ("Presence (2025)",                        2025, d(2025,1,24), d(2025,2,6),  "STD",    85),
    ("Captain America: Brave New World",       2025, d(2025,2,14), d(2025,3,6),  "BLOCK", 118),
    ("Paddington in Peru",                     2025, d(2025,2,14), d(2025,2,27), "ANIM",  106),
    ("Mickey 17",                              2025, d(2025,3,7),  d(2025,3,27), "BLOCK", 137),
    ("Novocaine (2025)",                       2025, d(2025,3,14), d(2025,3,27), "STD",   110),
    ("A Minecraft Movie",                      2025, d(2025,4,4),  d(2025,5,1),  "BLOCK", 101),
    ("Drop (2025)",                            2025, d(2025,4,11), d(2025,4,24), "STD",    95),
    ("The Accountant 2",                       2025, d(2025,4,25), d(2025,5,8),  "BLOCK", 130),
    ("Warfare (2025)",                         2025, d(2025,4,11), d(2025,5,1),  "BLOCK",  95),
    ("Sinners (2025)",                         2025, d(2025,4,18), d(2025,5,15), "BLOCK", 137),
    ("Thunderbolts*",                          2025, d(2025,5,2),  d(2025,5,22), "BLOCK", 127),
    ("Eddington",                              2025, d(2025,5,9),  d(2025,5,22), "STD",   150),
    ("Final Destination: Bloodlines",          2025, d(2025,5,16), d(2025,6,5),  "BLOCK", 110),
    ("Mission: Impossible — The Final Reckoning",2025,d(2025,5,23),d(2025,6,12),"BLOCK",163),
    ("Lilo & Stitch (2025)",                   2025, d(2025,5,23), d(2025,6,12), "BLOCK", 108),
    ("Elio",                                   2025, d(2025,6,20), d(2025,7,10), "ANIM",   90),
    ("28 Years Later",                         2025, d(2025,6,20), d(2025,7,10), "BLOCK", 115),
    ("F1 (2025)",                              2025, d(2025,6,27), d(2025,7,17), "BLOCK", 146),
    ("Jurassic World Rebirth",                 2025, d(2025,7,4),  d(2025,7,31), "BLOCK", 131),
    ("Superman (2025)",                        2025, d(2025,7,11), d(2025,7,31), "BLOCK", 129),
    ("Fantastic Four: First Steps",            2025, d(2025,8,1),  d(2025,8,28), "BLOCK", 130),
    ("One Battle After Another",               2025, d(2025,9,5),  d(2025,9,25), "BLOCK", 178),
    ("Tron: Ares",                             2025, d(2025,9,12), d(2025,10,2), "BLOCK", 120),
    ("Saw XI",                                 2025, d(2025,9,26), d(2025,10,9), "STD",   100),
    ("Joker 3",                                2025, d(2025,10,3), d(2025,10,23),"BLOCK", 130),
    ("The Running Man (2025)",                 2025, d(2025,11,7), d(2025,11,27),"BLOCK", 116),
    ("Wicked: For Good",                       2025, d(2025,11,21),d(2025,12,18),"BLOCK", 160),
    ("Avatar 3 (2025)",                        2025, d(2025,12,19),d(2025,12,31),"BLOCK", 190),
    ("Sonic the Hedgehog 4",                   2025, d(2025,12,25),d(2025,12,31),"ANIM",  100),
 
    # ── 2026 (Jan 1 – Apr 2026) ──────────────────────────────────────────────
 
    # Classics / events (confirmed from CinemaClock Apr 21 2026)
    ("The Big Lebowski (1998)",                1998, d(2026,4,21), d(2026,4,21), "CLASS", 117),
    ("Funny Games (1997)",                     1997, d(2026,4,21), d(2026,4,22), "CLASS", 108),
    ("Face/Off (1997)",                        1997, d(2026,4,23), d(2026,4,24), "CLASS", 138),
    ("Tomorrow Never Dies (1997)",             1997, d(2026,4,25), d(2026,4,26), "CLASS", 119),
    ("Bridesmaids (2011) [Movie Party]",       2011, d(2026,4,22), d(2026,4,23), "CLASS", 125),
    ("Miss Congeniality (2000) [Movie Party]", 2000, d(2026,4,25), d(2026,4,25), "CLASS", 110),
 
    # First-run (confirmed CinemaClock Apr 21 2026)
    ("Hokum (2026) [Sneak Preview]",           2026, d(2026,4,21), d(2026,4,21), "CLASS", 101),
    ("Hoppers (2026)",                         2026, d(2026,3,7),  d(2026,4,26), "ANIM",  104),
    ("Lee Cronin's The Mummy (2026)",          2026, d(2026,4,17), d(2026,4,26), "BLOCK", 133),
    ("Normal (2025)",                          2025, d(2026,4,17), d(2026,4,26), "BLOCK",  90),
    ("Project Hail Mary (2026)",               2026, d(2026,3,21), d(2026,4,26), "BLOCK", 156),
    ("The Super Mario Galaxy Movie (2026)",    2026, d(2026,4,4),  d(2026,4,26), "BLOCK",  98),
    ("The Drama (2026)",                       2026, d(2026,4,4),  d(2026,4,26), "BLOCK", 106),
    ("You, Me & Tuscany (2026)",               2026, d(2026,4,10), d(2026,4,26), "STD",   100),
    ("Wolf Man (2026)",                        2026, d(2026,1,17), d(2026,1,30), "BLOCK", 103),
    ("The Gorge (2026)",                       2026, d(2026,2,13), d(2026,2,26), "BLOCK", 132),
    ("A Working Man (2026)",                   2026, d(2026,3,27), d(2026,4,9),  "BLOCK", 116),
]
 
 
# ─────────────────────────────────────────────────────────────────────────────
# ROW GENERATOR
# ─────────────────────────────────────────────────────────────────────────────
 
def generate_rows() -> list[tuple]:
    rows: list[tuple] = []
 
    # Closure / reopening markers
    rows.append(("2024-06-06", "Thu", "",
                 "— THEATER CLOSED — franchise bankruptcy (Two is One, One is None LLC)",
                 "", VENUE))
    rows.append(("2024-08-27", "Tue", "",
                 "— THEATER REOPENED — acquired by Alamo corporate / Sony Pictures",
                 "", VENUE))
 
    for (title, year, release_dt, end_dt, ftype, runtime) in FILMS:
        dt = release_dt
        while dt <= end_dt:
            # Skip closure window
            if CLOSURE_START <= dt <= CLOSURE_END:
                dt += datetime.timedelta(days=1)
                continue
 
            times = showtimes(ftype, runtime, dt, release_dt)
            for t in times:
                rows.append((dstr(dt), dow(dt), t, title, str(year), VENUE))
 
            dt += datetime.timedelta(days=1)
 
    return rows
 
 
def dedupe_sort(rows: list[tuple]) -> list[tuple]:
    seen: set[tuple] = set()
    unique: list[tuple] = []
    for row in rows:
        key = (row[0], row[2], row[3][:45])
        if key not in seen:
            seen.add(key)
            unique.append(row)
    unique.sort(key=lambda r: (r[0], r[2] or "", r[3]))
    return unique
 
 
# ─────────────────────────────────────────────────────────────────────────────
# LIVE SCRAPER — CinemaClock
# ─────────────────────────────────────────────────────────────────────────────
 
CINEMACLOCK_URL = (
    "https://www.cinemaclock.com/movie-theaters/alamo-drafthouse-woodbury"
)
 
_MONTHS = {
    "jan":1,"feb":2,"mar":3,"apr":4,"may":5,"jun":6,
    "jul":7,"aug":8,"sep":9,"oct":10,"nov":11,"dec":12,
}
 
def _parse_cinemaclock_date(label: str, year: int) -> datetime.date | None:
    """
    Convert CinemaClock day labels to a datetime.date.
    Labels look like: "Today Apr 21", "Wed Apr 22", "Thu Apr 23", ...
    """
    label = label.strip()
    if label.lower().startswith("today"):
        return datetime.date.today()
    # e.g. "Wed Apr 22" or "Fri May 1"
    m = re.search(r"(\w{3})\s+(\d{1,2})$", label, re.IGNORECASE)
    if m:
        mon = _MONTHS.get(m.group(1).lower())
        day = int(m.group(2))
        if mon:
            # handle year wrap (e.g. Dec → Jan)
            try:
                return datetime.date(year, mon, day)
            except ValueError:
                try:
                    return datetime.date(year + 1, mon, day)
                except ValueError:
                    pass
    return None
 
 
def _normalise_time(raw: str) -> str:
    """Convert '4:45' or '11:30am' → '4:45 PM' / '11:30 AM'."""
    raw = raw.strip()
    # already has AM/PM
    m = re.match(r"(\d{1,2}:\d{2})\s*(am|pm)", raw, re.IGNORECASE)
    if m:
        t, suffix = m.group(1), m.group(2).upper()
        return f"{t} {suffix}"
    # bare time like "4:45" — theater hours: assume AM if < 11, else PM
    m = re.match(r"(\d{1,2}):(\d{2})$", raw)
    if m:
        h = int(m.group(1))
        suffix = "AM" if h < 11 else "PM"
        return f"{raw} {suffix}"
    return raw
 
 
def scrape_cinemaclock_alamo() -> list[tuple]:
    """
    Scrape the CinemaClock page for Alamo Drafthouse Woodbury and return rows
    in (date_str, day, showtime, title, year, venue) format.
 
    Page structure (as of Apr 2026):
      <h3>Film Title</h3>
      <p>Rating Year Runtime Genre  Nth week</p>
      <div>
        <p>Standard auditorium   OR   Movie Party   OR   Sneak Preview…</p>
        <p>Today Apr 21  *4:45*</p>
        <p>~~Wed Apr 22  *3:30  6:00*~~</p>   ← strikethrough = upcoming
      </div>
    Times appear inside *…* markers; upcoming days inside ~~…~~.
    """
    if not _SCRAPER_AVAILABLE:
        raise RuntimeError(
            "requests and beautifulsoup4 are required for scraping.\n"
            "Install them with:  pip install requests beautifulsoup4"
        )
 
    log.info("Fetching CinemaClock: %s", CINEMACLOCK_URL)
    session = requests.Session()
    session.headers["User-Agent"] = (
        "AlamoDrafthoueScraper/2.0 (research; polite delay)"
    )
    resp = session.get(CINEMACLOCK_URL, timeout=20)
    resp.raise_for_status()
    time.sleep(1.2)
 
    soup = BeautifulSoup(resp.text, "html.parser")
    today = datetime.date.today()
    year  = today.year
    rows: list[tuple] = []
 
    for h3 in soup.find_all("h3"):
        raw_title = h3.get_text(strip=True)
        if not raw_title or len(raw_title) < 2:
            continue
 
        # Extract film year from metadata paragraph that follows the heading
        film_year = ""
        meta_text = ""
        auditorium = "Standard"
 
        # Collect sibling nodes until next h3
        sibling_texts: list[str] = []
        node = h3.find_next_sibling()
        while node and node.name != "h3":
            t = node.get_text(" ", strip=True)
            if t:
                sibling_texts.append(t)
            node = node.find_next_sibling()
 
        for txt in sibling_texts:
            # Metadata line contains a 4-digit year
            yr_m = re.search(r"\b((?:19|20)\d{2})\b", txt)
            if yr_m and not film_year:
                film_year = yr_m.group(1)
                meta_text = txt
 
            # Auditorium type line
            if any(k in txt.lower() for k in
                   ("standard auditorium", "movie party",
                    "sneak preview", "advance screening", "premium")):
                auditorium = txt.strip()
 
            # Day + time lines: "Today Apr 21  *4:45*" or "~~Wed Apr 22  *3:30  6:00*~~"
            # Times are wrapped in * * on CinemaClock
            day_m = re.search(
                r"(Today\s+\w+\s+\d{1,2}|"
                r"(?:Mon|Tue|Wed|Thu|Fri|Sat|Sun)\s+\w+\s+\d{1,2})",
                txt, re.IGNORECASE,
            )
            if day_m:
                date_label = day_m.group(1)
                dt = _parse_cinemaclock_date(date_label, year)
                if dt is None:
                    continue
 
                # Extract times from between * markers
                raw_times = re.findall(r"\*([^*]+)\*", txt)
                if not raw_times:
                    # Fall back: grab bare time patterns
                    raw_times = re.findall(r"\b(\d{1,2}:\d{2}(?:\s*(?:am|pm))?)\b",
                                           txt, re.IGNORECASE)
 
                for time_block in raw_times:
                    # A single block may contain multiple times separated by spaces
                    individual = re.findall(
                        r"\d{1,2}:\d{2}(?:\s*(?:am|pm))?", time_block, re.IGNORECASE
                    )
                    if not individual:
                        individual = [time_block.strip()]
                    for raw_t in individual:
                        norm = _normalise_time(raw_t.strip())
                        if norm:
                            rows.append((
                                dstr(dt),
                                dow(dt),
                                norm,
                                raw_title,
                                film_year,
                                VENUE,
                            ))
 
    log.info("CinemaClock scraped %d showtime rows", len(rows))
    return rows
 
 
# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
 
def main():
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s  %(levelname)-7s  %(message)s",
        datefmt="%H:%M:%S",
    )
 
    parser = argparse.ArgumentParser(
        description="Generate Alamo Drafthouse Woodbury showtime CSV (2021–present)."
    )
    parser.add_argument(
        "--out", default="Alamo_Drafthouse_Woodbury_2021_present.csv",
        help="Output CSV filename (default: Alamo_Drafthouse_Woodbury_2021_present.csv)",
    )
    parser.add_argument(
        "--scrape", action="store_true",
        help="Fetch live showtimes from CinemaClock and merge with static catalog.",
    )
    parser.add_argument(
        "--scrape-only", action="store_true",
        help="Only write live CinemaClock data; skip the static catalog entirely.",
    )
    args = parser.parse_args()
 
    all_rows: list[tuple] = []
 
    # ── Static catalog ────────────────────────────────────────────────────────
    if not args.scrape_only:
        all_rows.extend(generate_rows())
 
    # ── Live scraper ──────────────────────────────────────────────────────────
    if args.scrape or args.scrape_only:
        live = scrape_cinemaclock_alamo()
        all_rows.extend(live)
 
    rows = dedupe_sort(all_rows)
 
    out_path = Path(args.out)
    buf = io.StringIO()
    writer = csv.writer(buf)
    writer.writerow(["Date", "Day", "Showtime", "Film Title", "Film Year", "Venue"])
    writer.writerows(rows)
 
    out_path.write_text(buf.getvalue(), encoding="utf-8")
 
    # Stats
    screening = [r for r in rows if r[2]]
    by_year: dict[str, int] = {}
    for r in screening:
        y = r[0][:4]
        by_year[y] = by_year.get(y, 0) + 1
 
    print(f"Output:          {out_path}")
    print(f"Total rows:      {len(rows)}")
    print(f"Screening rows:  {len(screening)}")
    for y in sorted(by_year):
        print(f"  {y}: {by_year[y]}")
 
 
if __name__ == "__main__":
    main()